
# MINI ML IMPLEMENTATION – Predictive Maintenance Failure Prediction

## Objective
Build a simple and explainable machine learning model to predict whether a machine operation will fail.

Dataset: `SciqusDS.csv`



## 1. Import Libraries


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier



## 2. Load Dataset


In [ ]:

df = pd.read_csv("SciqusDS.csv")

print("Dataset Shape:", df.shape)
df.head()



## 3. Understanding the Dataset

### Columns:
- `UDI` → Unique row identifier
- `Product ID` → Product identifier
- `Type` → Machine/Product type (categorical)
- Temperature, rotational speed, torque, tool wear → Numerical machine measurements
- `Target` → Failure prediction target (0 = No Failure, 1 = Failure)
- `Failure Type` → Text description of failure

### Goal:
Predict the `Target` column.


In [ ]:

df.info()


In [ ]:

df.isnull().sum()



## 4. Data Preprocessing

### Steps:
1. Remove identifier columns (`UDI`, `Product ID`)
2. Remove `Failure Type` because it directly reveals failure information (data leakage)
3. One-hot encode categorical column `Type`
4. Split data into training and testing sets


In [ ]:

X = df.drop(columns=["UDI", "Product ID", "Failure Type", "Target"])
y = df["Target"]

categorical_features = ["Type"]
numerical_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)



## 5. Model Selection

Two models are used:
1. Logistic Regression → Simple and interpretable baseline
2. Random Forest → Handles nonlinear patterns well and performs strongly on tabular data


In [ ]:

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_pipeline.fit(X_train, y_train)

y_pred_log = logistic_pipeline.predict(X_test)
y_prob_log = logistic_pipeline.predict_proba(X_test)[:,1]


In [ ]:

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:,1]



## 6. Evaluation Function


In [ ]:

def evaluate_model(name, y_true, y_pred, y_prob):
    print(f"\n{name}")
    print("-" * 40)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_prob)

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4))

    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(name)
    plt.show()



## 7. Logistic Regression Results


In [ ]:
evaluate_model('Logistic Regression', y_test, y_pred_log, y_prob_log)


## 8. Random Forest Results


In [ ]:
evaluate_model('Random Forest', y_test, y_pred_rf, y_prob_rf)


## 9. Conclusion

### Observations:
- Random Forest generally performs better because it captures complex relationships between machine measurements and failures.
- Logistic Regression is simpler and easier to interpret.
- Removing `Failure Type` was important to avoid data leakage.

### Final Choice:
Random Forest is selected as the better model for this task due to higher predictive performance.

### Improvements if More Time Was Available:
- Hyperparameter tuning
- Cross-validation
- Feature importance analysis
- Handling class imbalance
- Trying XGBoost or LightGBM
